In [ ]:

# 1. Load Dataset

import pandas as pd

df = pd.read_csv(
    "../datasets/CMAPSSData/train_FD001.txt",
    sep=r"\s+",
    header=None
)

print("Dataset Shape:", df.shape)



# 2. Create RUL (Remaining Useful Life)
max_cycles = df.groupby(0)[1].max()

df['RUL'] = df.apply(
    lambda row: max_cycles[row[0]] - row[1],
    axis=1
)

print(df[[0, 1, 'RUL']].head())



# 3. Remove Constant Columns

constant_cols = [col for col in df.columns if df[col].nunique() == 1]

print("Constant Columns:", constant_cols)

df_clean = df.drop(columns=constant_cols)

print("Clean Dataset Shape:", df_clean.shape)



# 4. Prepare Features and Target

X = df_clean.drop(columns=['RUL', 0])  # Remove RUL and Engine ID
y = df_clean['RUL']

print("X Shape:", X.shape)
print("y Shape:", y.shape)



# 5. Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# 6. Feature Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "scaler.pkl")
print("Scaler saved successfully!")

print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape :", X_test_scaled.shape)



# 7. Train Linear Regression Model

from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_scaled, y_train)

print("Model Training Completed")


# 8. Make Predictions
y_pred = model.predict(X_test_scaled)

print("Predicted:", y_pred[:5])
print("Actual   :", y_test[:5].values)



# 9. Evaluate Model

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== Model Evaluation =====")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


#Random Forest model
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)
print("Random Forest Training Completed....")

rf_pred = rf_model.predict(X_test_scaled)
print("RF Predicted:",rf_pred[:5])
print("Actual  :",y_test[:5].values)

rf_pred = rf_model.predict(X_test_scaled)

rf_mae = mean_absolute_error(y_test,rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test,rf_pred))
rf_r2 = r2_score(y_test,rf_pred)

print("RF MAE :",rf_mae)
print("RF RMSE :",rf_rmse)
print("RF R² :",rf_r2)

train_pred = rf_model.predict(X_train_scaled)

train_mae = mean_absolute_error(y_train, train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
train_r2 = r2_score(y_train, train_pred)


print("Train MAE :",train_mae)
print("Train RMSE :", train_rmse)
print("Train R² :", train_r2)


#load the random forest model 
import joblib
joblib.dump(rf_model,"random_forest_rul.pkl")

#First Hyperparameter tuing 
rf_model2 = RandomForestRegressor(
    n_estimators = 100,
    max_depth = 10,
    random_state = 42
)

rf_model2.fit(X_train_scaled, y_train)

rf2_pred = rf_model2.predict(X_test_scaled)

rf2_mae = mean_absolute_error(y_test, rf2_pred)
rf2_rmse = np.sqrt(mean_squared_error(y_test,rf2_pred))
rf2_r2 = r2_score(y_test, rf2_pred)

print("RF2 MAE :",rf2_mae)
print("RF2 RMSE :",rf2_rmse)
print("RF2 R² :",rf2_r2)

train_pred2 = rf_model2.predict(X_train_scaled)
train_r2_2 = r2_score(y_train, train_pred2)
print("Train R² :",train_r2_2)

#importing XGBOOSt model
import xgboost
#print(xgboost.__version__)

from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators = 100,
    random_state = 42
)

xgb_model.fit(X_train_scaled,y_train)
print("XGBOOST Training Completed")

xgb_pred = xgb_model.predict(X_test_scaled)

xgb_mae = mean_absolute_error(y_test,xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test,xgb_pred))
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGB MAE :", xgb_mae)
print("XGB RMSE :", xgb_rmse)
print("XGB R² :", xgb_r2)

xgb_train_pred = xgb_model.predict(X_train_scaled)
xgb_train_r2 = r2_score(y_train, xgb_train_pred)
print("XGB Train R² :",xgb_train_r2)


#Tuning parameters of XGBoost model


xgb_model2 = XGBRegressor(
    n_estimators = 200,
    learning_rate = 0.1,
    max_depth = 5,
    random_state = 42
)

xgb_model2.fit(X_train_scaled, y_train)
print("XGBoost Training Completed ")

xgb2_pred = xgb_model2.predict(X_test_scaled)

xgb2_mae = mean_absolute_error(y_test, xgb2_pred)
xgb2_rmse = np.sqrt(mean_squared_error(y_test,xgb2_pred))
xgb2_r2 = r2_score(y_test,xgb2_pred)

print("XGB2 MAE :",xgb2_mae)
print("XGB2 RMSE :", xgb2_rmse)
print("XGB2  R² ", xgb_r2)

xgb2_train_pred = xgb_model2.predict(X_train_scaled)
xgb2_train_r2 = r2_score(y_train, xgb2_train_pred)

print("XGB2 Train  R² :", xgb2_train_r2)

   
df_clean.head()


import numpy as np 

def create_sequences(data, sequence_length):
    X_seq = []
    y_seq = []
    
    for i in range(len(data)- sequence_length + 1):
        X_seq.append(
            data.iloc[i:i+sequence_length]
            .drop(columns=['RUL'])
            .values
        )
        
        y_seq.append(
            data.iloc[i+sequence_length-1]['RUL']
            
        )
        
    return np.array(X_seq),np.array(y_seq)
sequence_length = 30

engine_1 = df_clean[df_clean[0]==1]

X_seq , y_seq = create_sequences(
    engine_1,
    sequence_length
)
print("X_seq Shape :", X_seq.shape)
print("y_seq Shape :", y_seq.shape)


all_X = []
all_y = []

sequence_length = 30

for engine_id in df_clean[0].unique():
    
    engine_data = df_clean[
        df_clean[0] == engine_id
    ]
    
    X_seq, y_seq = create_sequences(
        engine_data,
        sequence_length
    )
    
    all_X.extend(X_seq)
    all_y.extend(y_seq)
    
    
X_lstm = np.array(all_X)
y_lstm = np.array(all_y)

print("X_lstm Shape: ", X_lstm.shape)
print("y_lstm.shape :", y_lstm.shape)

#Isolation Forest
from sklearn.ensemble import IsolationForest
import joblib

# Use the same scaled features you already prepared
iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

iso_model.fit(X_train_scaled)

print("Isolation Forest Training Completed!")

# Save the model
joblib.dump(iso_model, "isolation_forest.pkl")

print("Isolation Forest Model Saved Successfully!")


#Train-test split

X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_lstm,
    y_lstm, 
    test_size = 0.2,
    random_state = 42
)

print("X_train_lstm :", X_train_lstm.shape)
print("X_test_lstm :", X_test_lstm.shape)
print("y_train_lstm :", y_train_lstm.shape)
print("y_test_lstm :", y_test_lstm.shape)


import tensorflow as tf

from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(
        50, 
        input_shape = (30,19)
        
    ),
    Dense(1)
])

model.compile(
    optimizer = 'adam',
    loss = 'mse',
    metrics = ['mae']
)
model.summary()

history = model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs = 10,
    batch_size = 32,
    validation_split = 0.2,
    verbose= 1
)

test_loss, test_mae = model.evaluate(
    X_test_lstm, 
    y_test_lstm, 
    verbose = 1
)
print("Test MAE : ", test_mae )

import joblib

joblib.dump(rf_model2, "random_forest_rul.pkl")
print("Model Saved Successfully ")

import os 
print(os.listdir())

print(df.columns.tolist())

In [ ]:
# Create failure label
failure_threshold = 30

df["Failure"] = (df["RUL"] <= failure_threshold).astype(int)

print(df["Failure"].value_counts())

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# -------------------------------
# Select the same 5 production features
# -------------------------------
sensor_columns = [6, 7, 11, 15, 18]

X_failure = df[sensor_columns]
y_failure = df["Failure"]

# -------------------------------
# Train-Test Split
# -------------------------------
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_failure,
    y_failure,
    test_size=0.2,
    random_state=42,
    stratify=y_failure
)

# -------------------------------
# Scale Data
# -------------------------------
failure_scaler = StandardScaler()

X_train_scaled = failure_scaler.fit_transform(X_train_f)
X_test_scaled = failure_scaler.transform(X_test_f)

# -------------------------------
# Train XGBoost Classifier
# -------------------------------
failure_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

failure_model.fit(X_train_scaled, y_train_f)

# -------------------------------
# Evaluation
# -------------------------------
predictions = failure_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test_f, predictions))
print(classification_report(y_test_f, predictions))

# -------------------------------
# Save Model
# -------------------------------
joblib.dump(failure_model, "failure_prediction.pkl")
joblib.dump(failure_scaler, "failure_scaler.pkl")

print("Failure Prediction Model Saved Successfully!")

Accuracy: 0.958565543978677
              precision    recall  f1-score   support

           0       0.97      0.98      0.98      3507
           1       0.88      0.84      0.86       620

    accuracy                           0.96      4127
   macro avg       0.93      0.91      0.92      4127
weighted avg       0.96      0.96      0.96      4127

Failure Prediction Model Saved Successfully!


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import joblib

# Five sensor features used by the application
sensor_columns = [6, 7, 11, 15, 18]

X_anomaly = df[sensor_columns]

# Scale
scaler_anomaly = StandardScaler()
X_scaled = scaler_anomaly.fit_transform(X_anomaly)

# Train Isolation Forest
iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

iso_model.fit(X_scaled)

# Save both
joblib.dump(iso_model, "isolation_forest.pkl")
joblib.dump(scaler_anomaly, "anomaly_scaler.pkl")

print("Isolation Forest retrained successfully!")
print("Anomaly scaler saved!")

🥈 Linear Regression

MAE  = 30.54
RMSE = 39.70
R²   = 0.655

🥇 Random Forest

MAE  = 25.45
RMSE = 35.92
R²   = 0.718